# 01 — Conhecendo indicadores de saúde: estatística descritiva e visualização

            [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flavioluizseixas/aprendizado-de-maquina-para-saude/blob/main/notebooks/01_estatistica_descritiva.ipynb)

            **Duração estimada:** 45–60 minutos  
            **Pré-requisitos:** Python inicial, pandas e leitura de gráficos.

            ## Objetivos

            - baixar e inspecionar uma base pública
- distinguir variáveis numéricas, binárias e ordinais
- resumir distribuições e escolher gráficos adequados
- discutir autorrelato, associação e limites de generalização

            ## Fonte e licença

            CDC Diabetes Health Indicators (BRFSS), obtida da UCI, conjunto 891.

            Consulte a licença CC BY 4.0 e a citação exibidas na página da UCI.

            > **Uso responsável:** Este material tem finalidade exclusivamente educacional. Os resultados não devem ser usados para diagnóstico, prognóstico, tratamento, gestão assistencial ou decisão de saúde pública sem validação adequada, análise de contexto e supervisão de profissionais qualificados.

## Preparação do ambiente

> Como tornar a execução repetível e sem upload manual?

In [ ]:
# Preparação reproduzível do ambiente (a instalação ocorre só se faltar pacote).
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO = "flavioluizseixas/aprendizado-de-maquina-para-saude"
REPO_DIR = Path("/content") / REPO.split("/")[-1]
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    command = ["git", "clone", f"https://github.com/{REPO}.git", str(REPO_DIR)]
    if REPO_DIR.exists():
        command = ["git", "-C", str(REPO_DIR), "pull", "--ff-only"]
    subprocess.run(command, check=True)
    os.chdir(REPO_DIR)
else:
    candidates = [Path.cwd(), Path.cwd().parent]
    project = next((p for p in candidates if (p / "src").exists()), Path.cwd())
    os.chdir(project)

packages = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.1,<4', 'matplotlib': 'matplotlib>=3.8,<4', 'seaborn': 'seaborn>=0.13,<1', 'sklearn': 'scikit-learn>=1.4,<2', 'requests': 'requests>=2.31,<3', 'ucimlrepo': 'ucimlrepo>=0.0.7,<1'}
missing = [spec for module, spec in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

from src.config import RANDOM_STATE, seed_everything
seed_everything(RANDOM_STATE)
print(f"Ambiente pronto em {Path.cwd()} | Colab={IN_COLAB} | semente={RANDOM_STATE}")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

from src.data_loading import load_cdc_diabetes
from src.visualization import plot_variable

FAST_MODE = True
SAMPLE_SIZE = 20_000 if FAST_MODE else None  # None preserva a base completa
sns.set_theme(style="whitegrid")

## Pergunta orientadora

> Como se distribuem indicadores autorrelatados de saúde nesta amostra, e quais conclusões eles não sustentam?

## Obtenção dos dados

> A fonte e a amostragem ficaram registradas?

In [ ]:
data, metadata = load_cdc_diabetes(SAMPLE_SIZE, random_state=RANDOM_STATE)
print({key: metadata.get(key) for key in ["name", "uci_id", "target", "sample_size"]})
print(metadata["feature_types"])

## Inspeção

> Qual é o tamanho, o tipo e a qualidade aparente da tabela?

In [ ]:
display(data.head())
print("Dimensão:", data.shape)
display(data.dtypes.rename("tipo").to_frame().T)
display(data.isna().sum().rename("ausências").to_frame().T)
print("Linhas duplicadas:", data.duplicated().sum())

In [ ]:
target_percent = data["Diabetes_binary"].value_counts(normalize=True).sort_index() * 100
display(target_percent.rename("percentual").round(1).to_frame())
target_percent.plot.bar(title=f"Indicação de diabetes/pré-diabetes (n={len(data):,})")
plt.ylabel("Percentual (%)")
plt.xlabel("Diabetes_binary")
plt.show()

### Como interpretar

A barra descreve a prevalência do **indicador-alvo na amostra**. Ela não mede o desempenho de um diagnóstico e não explica por que o desfecho ocorreu.

## Preparação

> Quais resumos são adequados para distribuições possivelmente assimétricas?

In [ ]:
selected = ["BMI", "Age", "GenHlth", "PhysHlth", "MentHlth", "HighBP", "Diabetes_binary"]
description = data[selected].describe(percentiles=[0.25, 0.5, 0.75]).T
description["IQR"] = description["75%"] - description["25%"]
display(description[["count", "mean", "std", "25%", "50%", "75%", "IQR"]].round(2))

## Experimento

> Qual gráfico responde melhor a cada tipo de variável?

In [ ]:
plot_variable(data, "BMI"); plt.show()

In [ ]:
plot_variable(data, "HighBP"); plt.show()

In [ ]:
plot_variable(data, "GenHlth", kind="ordinal"); plt.show()

### Como interpretar

Para BMI, mediana e IQR são resistentes a extremos. Para variáveis binárias e ordinais, percentuais preservam uma leitura mais direta. Diferenças entre grupos são associações descritivas, não efeitos causais.

## Avaliação visual conjunta

> É possível observar relações sem desenhar 250 mil pontos?

In [ ]:
pair_columns = ["BMI", "Age", "GenHlth", "PhysHlth", "MentHlth", "Diabetes_binary"]
pair_n = min(1_000, len(data) - 1)
pair_sample, _ = train_test_split(
    data[pair_columns], train_size=pair_n,
    stratify=data["Diabetes_binary"], random_state=RANDOM_STATE,
)
sns.pairplot(pair_sample, hue="Diabetes_binary", corner=True, plot_kws={"alpha": 0.35, "s": 18})
plt.show()

### Como interpretar

A amostra estratificada de até 1.000 linhas torna o `pairplot` legível e rápido, preservando aproximadamente as classes. Sobreposição indica que um atributo isolado dificilmente separa perfeitamente os grupos.

## Limitações e responsabilidade

- Os indicadores do BRFSS incluem autorrelato, sujeito a memória e classificação incorreta.
- Amostragem, representação de grupos e desbalanceamento limitam a generalização.
- Uma associação visual não estabelece causalidade nem substitui avaliação clínica.

## Atividade

Escolha outro atributo, use `plot_variable` e escreva: (1) o que o gráfico sugere nesta amostra; (2) o que não pode ser concluído.

## Três aprendizados principais

1. O tipo da variável orienta o resumo e o gráfico.
2. Mediana e IQR ajudam em distribuições assimétricas.
3. Visualização descreve a amostra, mas não prova causalidade.

## Referências

- [UCI — CDC Diabetes Health Indicators](https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators)
- [CDC — BRFSS](https://www.cdc.gov/brfss/)

## Versões das bibliotecas

Registre o ambiente junto ao resultado.

In [ ]:
from src.config import library_versions
library_versions(('numpy', 'pandas', 'matplotlib', 'seaborn', 'scikit-learn', 'ucimlrepo'))